In [3]:
def read_graph(filename):
    with open(filename, "r") as f:
        n = int(f.readline().strip())
        m = int(f.readline().strip())

        graph = [[] for _ in range(n + 1)]  # graph[0] inutilisé

        for _ in range(n):
            parts = f.readline().split()
            u = int(parts[0])
            nb_neighbors = int(parts[1])

            idx = 2
            for _ in range(nb_neighbors):
                v = int(parts[idx])
                graph[u].append(v)
                idx += 2  # on saute aussi le poids

    return graph


def compute_in_degrees_from_graph(graph):
    n = len(graph) - 1
    in_degree = [0] * (n + 1)

    for u in range(1, n + 1):
        for v in graph[u]:
            in_degree[v] += 1

    return in_degree

In [4]:
import random
import bisect

def choose_preferential_targets(in_degree, nb_links):
    n = len(in_degree) - 1

    if nb_links > n:
        raise ValueError("nb_links ne peut pas dépasser le nombre de sommets existants")

    cumulative = []
    total = 0

    for i in range(1, n + 1):
        total += in_degree[i] + 1
        cumulative.append(total)

    targets = set()

    while len(targets) < nb_links:
        r = random.uniform(0, total)
        idx = bisect.bisect_left(cumulative, r)
        targets.add(idx + 1)

    return list(targets)

In [18]:
def add_preferential_nodes(graph, nb_new_nodes, nb_links_per_new_node):
    old_n = len(graph) - 1
    in_degree = compute_in_degrees_from_graph(graph)

    for new_node in range(old_n + 1, old_n + nb_new_nodes + 1):
        targets = choose_preferential_targets(in_degree, nb_links_per_new_node)

        graph.append(targets)

        # le nouveau sommet a pour l’instant degré entrant 0
        in_degree.append(0)

        # les anciennes cibles gagnent un lien entrant
        for v in targets:
            in_degree[v] += 1

    return graph

def add_isolated_nodes(graph, nb_new_nodes):
    for _ in range(nb_new_nodes):
        graph.append([])

    return graph

In [6]:
def write_graph(graph, filename):
    n = len(graph) - 1
    m = sum(len(graph[u]) for u in range(1, n + 1))

    with open(filename, "w") as f:
        f.write(f"{n}\n")
        f.write(f"{m}\n")

        for u in range(1, n + 1):
            neighbors = graph[u]
            degree = len(neighbors)

            f.write(f"{u} {degree}")

            if degree > 0:
                weight = 1.0 / degree
                for v in neighbors:
                    f.write(f" {v} {weight:.12f}")

            f.write("\n")

In [19]:
random.seed(0)
dirname = "C:\\Users\\Neil Bonnard\\OneDrive\\Documents\\UVSQ\\Semestre 2 2025-2026\\Ranking\\Projet Ranking\\Petites matrices\\G8.txt"
graph = read_graph(dirname)

new_graph = add_preferential_nodes(
    graph,
    nb_new_nodes=3,
    nb_links_per_new_node=2
)

write_graph(new_graph, "G8_preferential-3-2.txt")

graph = read_graph(dirname)

new_graph = add_isolated_nodes(graph, 3)

write_graph(new_graph, "G8_isolated.txt")



In [8]:
dirname = "C:\\Users\\Neil Bonnard\\OneDrive\\Documents\\UVSQ\\Semestre 2 2025-2026\\Ranking\\Projet Ranking\\Petites matrices\\G8.txt"
graph = read_graph(dirname)
in_degree = compute_in_degrees_from_graph(graph)
print(in_degree)
targets = choose_preferential_targets(in_degree, 3)
print("Cibles choisies :", targets)

[0, 2, 1, 2, 2, 1, 2, 1, 1]
Cibles choisies : [3, 4, 6]


In [9]:
import subprocess
import re
from pathlib import Path

C_FILE = Path("pagerank.c")
EXE = Path("pagerank.exe")  # Windows

def compile_pagerank():
    cmd = ["gcc", "-O3", str(C_FILE), "-o", str(EXE)]
    subprocess.run(cmd, check=True)

def run_pagerank(graph_file, alpha=0.85, epsilon=1e-6):
    cmd = [
        str(EXE),
        str(graph_file),
        str(alpha),
        str(epsilon)
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
        check=True
    )

    output = result.stdout

    match = re.search(r"Converged in (\d+) iterations", output)
    if match is None:
        raise ValueError("Impossible de trouver le nombre d'itérations dans la sortie C")

    iterations = int(match.group(1))

    return iterations, output

In [16]:
!gcc pagerank.c -O3 -o pagerank.exe
!.\pagerank.exe G8_preferential.txt 0.85 1e-6

Graph loaded: N=11 nodes, M=18 arcs
Alpha=0.8500, Epsilon=1.00e-006
Converged in 51 iterations

First 10 PageRank scores:
  node 1 : 0.15185748
  node 2 : 0.07817570
  node 3 : 0.12047069
  node 4 : 0.18440273
  node 5 : 0.08588126
  node 6 : 0.13430254
  node 7 : 0.13358883
  node 8 : 0.07041169
  node 9 : 0.01363636
  node 10 : 0.01363636


In [26]:
alphas = [0.5, 0.7, 0.85, 0.9, 0.99]

results = []
results2 = []
for alpha in alphas:
    iters, _ = run_pagerank("G8_preferential.txt", alpha=alpha, epsilon=1e-6)
    results.append((alpha, iters))

print(results)

for alpha in alphas:
    iters, _ = run_pagerank("G8_isolated.txt", alpha=alpha, epsilon=1e-6)
    results2.append((alpha, iters))

print(results2)

[(0.5, 16), (0.7, 28), (0.85, 51), (0.9, 67), (0.99, 149)]
[(0.5, 15), (0.7, 27), (0.85, 49), (0.9, 65), (0.99, 144)]
